In [23]:
import simpy
import random

#parameters
num_of_households = 10
sms_response_probability = 0.3
SEED = 1
rng = random.Random(SEED)
sms_timing = 10  # day when SMS is sent for send_sms_on_day_n function

intervention_costs = {
    "sms": 0.05,
}

In [24]:
class Household:
    def __init__(self, id, attribute1, attribute2):
        self.id = id
        self.attribute1 = attribute1
        self.attribute2 = attribute2
        # Add more attributes as needed

        self.responded = rng.choice([True, False])  # this would be based on the attributes in the future, but for now it's random

#hcange how the household is represented when printed
    def __repr__(self):
        return (
            f"Household(id={self.id}, attribute1={self.attribute1}, "
            f"attribute2={self.attribute2}, responded={self.responded})"
        )

In [25]:
def create_cost_tracker():
    return {
        "total_cost": 0.0,
        "intervention_counts": {},
        "intervention_cost_totals": {},
    }


def record_intervention_cost(cost_tracker, intervention_type, units=1):
    unit_cost = intervention_costs.get(intervention_type, 0.0)
    total_cost = unit_cost * units

    cost_tracker["intervention_counts"][intervention_type] = (
        cost_tracker["intervention_counts"].get(intervention_type, 0) + units
    )
    cost_tracker["intervention_cost_totals"][intervention_type] = (
        cost_tracker["intervention_cost_totals"].get(intervention_type, 0.0) + total_cost
    )
    cost_tracker["total_cost"] += total_cost


def summarise_costs(cost_tracker):
    return {
        "total_cost": round(cost_tracker["total_cost"], 2),
        "intervention_counts": cost_tracker["intervention_counts"],
        "intervention_cost_totals": {
            intervention_type: round(total_cost, 2)
            for intervention_type, total_cost in cost_tracker["intervention_cost_totals"].items()
        },
    }


def send_sms_on_day_n(env, households, cost_tracker):
    yield env.timeout(sms_timing)

    print(f"Day {env.now}: sending SMS reminders")
    for household in households:
        if household.responded is False:
            print(f"SMS sent to household {household.id} on day {env.now}")
            record_intervention_cost(cost_tracker, "sms")
            if rng.random() < sms_response_probability:
                household.responded = True
                print(f"Household {household.id} responded after SMS")


In [26]:
def send_sms_every_day(env, households, cost_tracker):
    while True:
        yield env.timeout(1)  # wait for 1 day
        for household in households:
            if household.responded is False:
                print(f"SMS sent to household {household.id} on day {env.now}")
                record_intervention_cost(cost_tracker, "sms")
                if rng.random() < sms_response_probability:  # again, this would be based on the attributes in the future, but for now it's a flat measure
                    household.responded = True
                    print(f"Household {household.id} responded after SMS")

In [27]:
def daily_responce_totals(env, households):
    response_list = []
    while True:
        yield env.timeout(1)
        response_list.append(sum(h.responded for h in households))
        print(f"on day {env.now}, the last 3 days of response are {response_list[-3:]}")


In [28]:
#setup the environment
env = simpy.Environment()
cost_tracker = create_cost_tracker()


#generate households, attributes range from 1 to 10
hh_list = [Household(id=i, attribute1=rng.randint(1, 10), attribute2=rng.randint(1, 10)) for i in range(num_of_households)]
for h in hh_list:
    print(h)

#processes, #out those that are not needed
##env.process(send_sms_on_day_n(env, hh_list, cost_tracker))
env.process(send_sms_every_day(env, hh_list, cost_tracker))

#run simulation
env.run(until = 100)

#print results
for i in hh_list:
    print(f"Household {i.id} responded: {i.responded}")

cost_summary = summarise_costs(cost_tracker)
print(f"Total intervention cost: {cost_summary['total_cost']:.2f}")
print(f"Intervention counts: {cost_summary['intervention_counts']}")
print(f"Intervention cost totals: {cost_summary['intervention_cost_totals']}")

Household(id=0, attribute1=3, attribute2=10, responded=True)
Household(id=1, attribute1=5, attribute2=2, responded=False)
Household(id=2, attribute1=8, attribute2=8, responded=False)
Household(id=3, attribute1=4, attribute2=2, responded=False)
Household(id=4, attribute1=1, attribute2=7, responded=False)
Household(id=5, attribute1=10, attribute2=1, responded=False)
Household(id=6, attribute1=5, attribute2=4, responded=True)
Household(id=7, attribute1=6, attribute2=1, responded=True)
Household(id=8, attribute1=1, attribute2=9, responded=True)
Household(id=9, attribute1=7, attribute2=4, responded=False)
SMS sent to household 1 on day 1
SMS sent to household 2 on day 1
SMS sent to household 3 on day 1
SMS sent to household 4 on day 1
SMS sent to household 5 on day 1
SMS sent to household 9 on day 1
SMS sent to household 1 on day 2
SMS sent to household 2 on day 2
SMS sent to household 3 on day 2
SMS sent to household 4 on day 2
SMS sent to household 5 on day 2
SMS sent to household 9 on da